In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    rawPredictionCol="lr_raw_prediction",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial",
    standardization=True,
    aggregationDepth=2
)

lr_model = lr.fit(train_df)
print(f" Model trained in {lr_model.summary.totalIterations} iterations")

In [ ]:
print("\n" + "-"*80)
print("PERFORMANCE METRICS")
print("-"*80)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    metricName="f1"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    metricName="weightedRecall"
)

# Calculate metrics
print("\n1. ACCURACY:")
train_accuracy_lr = evaluator_acc.evaluate(train_pred_lr)
val_accuracy_lr = evaluator_acc.evaluate(val_pred_lr)
test_accuracy_lr = evaluator_acc.evaluate(test_pred_lr)
print(f"   Training:   {train_accuracy_lr:.4f}")
print(f"   Validation: {val_accuracy_lr:.4f}")
print(f"   Test:       {test_accuracy_lr:.4f}")

print("\n2. F1 SCORE:")
train_f1_lr = evaluator_f1.evaluate(train_pred_lr)
val_f1_lr = evaluator_f1.evaluate(val_pred_lr)
test_f1_lr = evaluator_f1.evaluate(test_pred_lr)
print(f"   Training:   {train_f1_lr:.4f}")
print(f"   Validation: {val_f1_lr:.4f}")
print(f"   Test:       {test_f1_lr:.4f}")

print("\n3. WEIGHTED PRECISION:")
train_precision_lr = evaluator_precision.evaluate(train_pred_lr)
val_precision_lr = evaluator_precision.evaluate(val_pred_lr)
test_precision_lr = evaluator_precision.evaluate(test_pred_lr)
print(f"   Training:   {train_precision_lr:.4f}")
print(f"   Validation: {val_precision_lr:.4f}")
print(f"   Test:       {test_precision_lr:.4f}")

print("\n4. WEIGHTED RECALL:")
train_recall_lr = evaluator_recall.evaluate(train_pred_lr)
val_recall_lr = evaluator_recall.evaluate(val_pred_lr)
test_recall_lr = evaluator_recall.evaluate(test_pred_lr)
print(f"   Training:   {train_recall_lr:.4f}")
print(f"   Validation: {val_recall_lr:.4f}")
print(f"   Test:       {test_recall_lr:.4f}")

In [ ]:
print("\n" + "-"*80)
print("CONFUSION MATRIX (Test Set)")
print("-"*80)

confusion_matrix = test_pred_lr.groupBy("length_category_idx", "lr_prediction").count()
confusion_matrix = confusion_matrix.orderBy("length_category_idx", "lr_prediction")
confusion_matrix.show(20)

confusion_pd = confusion_matrix.toPandas()
confusion_pivot = confusion_pd.pivot(
    index='length_category_idx',
    columns='lr_prediction',
    values='count'
).fillna(0).astype(int)

print("\nConfusion Matrix:")
print(confusion_pivot)
confusion_pivot.to_csv('lr_confusion_matrix.csv')
print(" Saved to 'lr_confusion_matrix.csv'")

In [ ]:
print("\n" + "-"*80)
print("PER-CLASS PERFORMANCE")
print("-"*80)

per_class_metrics = []
for class_idx in range(int(test_pred_lr.agg({"length_category_idx": "max"}).collect()[0][0]) + 1):
    tp = test_pred_lr.filter((col("length_category_idx") == class_idx) & (col("lr_prediction") == class_idx)).count()
    fp = test_pred_lr.filter((col("length_category_idx") != class_idx) & (col("lr_prediction") == class_idx)).count()
    fn = test_pred_lr.filter((col("length_category_idx") == class_idx) & (col("lr_prediction") != class_idx)).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    class_name = test_pred_lr.filter(col("length_category_idx") == class_idx).select("length_category").first()[0]

    per_class_metrics.append({
        'class_idx': class_idx,
        'class_name': class_name,
        'support': tp + fn,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    })

per_class_df = pd.DataFrame(per_class_metrics)
print("\n" + per_class_df.to_string(index=False))
per_class_df.to_csv('lr_per_class_metrics.csv', index=False)
print("\n Saved to 'lr_per_class_metrics.csv'")

In [ ]:
print("\n" + "-"*80)
print("FEATURE IMPORTANCE")
print("-"*80)

coefficients = lr_model.coefficientMatrix.toArray()
feature_importance_lr = np.abs(coefficients).mean(axis=0)

feature_importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': feature_importance_lr
}).sort_values('importance', ascending=False)

print("\nTop 10 Features:")
print(feature_importance_df.head(10).to_string(index=False))
feature_importance_df.to_csv('lr_feature_importance.csv', index=False)
print(" Saved to 'lr_feature_importance.csv'")

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON: LR vs RF")
print("="*80)

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Training Time'],
    'Logistic Regression': [
        f"{test_accuracy_lr:.4f}",
        f"{test_f1_lr:.4f}",
        f"{test_precision_lr:.4f}",
        f"{test_recall_lr:.4f}",
        f"{time.time() - start_time:.2f}s"
    ],
    'Random Forest': [
        f"{test_accuracy:.4f}",
        f"{test_f1:.4f}",
        "N/A",
        "N/A",
        f"{rf_time:.2f}s"
    ]
})

print("\n" + comparison.to_string(index=False))
comparison.to_csv('lr_vs_rf_comparison.csv', index=False)